# Colab test B — pycaret-core 3.5.0 (community fork)

Expected: install succeeds without touching numpy/pandas/scikit-learn on Python 3.13 (no restart), all notebook API calls work unchanged, and the lightgbm / xgboost / catboost forecasters train.

**Run this in a fresh Colab runtime** (Runtime → Disconnect and delete runtime, then Run all).

## 1. Runtime before install

In [1]:
import sys, importlib.metadata as md
print("Python:", sys.version.split()[0])
for p in ["numpy","pandas","scikit-learn","scipy","statsmodels","sktime","matplotlib","lightgbm","xgboost","catboost","pycaret","pycaret-core"]:
    try: print(f"  {p:14s} {md.version(p)}")
    except md.PackageNotFoundError: print(f"  {p:14s} (not installed)")

Python: 3.13.15
  numpy          2.1.3
  pandas         2.2.3
  scikit-learn   1.6.1
  scipy          1.16.3
  statsmodels    0.15.0
  sktime         (not installed)
  matplotlib     3.10.0
  lightgbm       4.6.0
  xgboost        3.4.1
  catboost       (not installed)
  pycaret        (not installed)
  pycaret-core   (not installed)


## 2. Install the fork

`statsmodels<0.15` is required until sktime ships the fix merged 2026-09-13 (sktime PR 10972).

In [2]:
!pip install -q pycaret-core "statsmodels<0.15" lightgbm xgboost catboost

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.8/106.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.9/107.9 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.7/82.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.9/688.9 kB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.8/418.8 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.6/37.6 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 24.3 MB/s eta 0:00:00
   

## 3. Runtime after install

If Colab shows a *Restart session* banner, restart and continue from here.

In [3]:
import sys, importlib.metadata as md
print("Python:", sys.version.split()[0])
for p in ["numpy","pandas","scikit-learn","scipy","statsmodels","sktime","matplotlib","lightgbm","xgboost","catboost","pycaret","pycaret-core"]:
    try: print(f"  {p:14s} {md.version(p)}")
    except md.PackageNotFoundError: print(f"  {p:14s} (not installed)")

Python: 3.13.15
  numpy          2.1.3
  pandas         2.2.3
  scikit-learn   1.6.1
  scipy          1.16.3
  statsmodels    0.14.6
  sktime         1.1.0
  matplotlib     3.10.0
  lightgbm       4.6.0
  xgboost        3.4.1
  catboost       1.2.10
  pycaret        (not installed)
  pycaret-core   3.5.0


## 4. The same API the course notebooks use

In [4]:
import warnings; warnings.filterwarnings("ignore")
import pycaret; print("pycaret.__version__ =", pycaret.__version__)
from pycaret.utils import version; print("pycaret.utils.version() =", version())
from pycaret.datasets import get_data
from pycaret.time_series import *

data = get_data("airline")
exp = TSForecastingExperiment()
exp.setup(data=data, fh=12, coverage=0.90, session_id=42)
print("models available:", len(exp.models()), "| catboost_cds_dt:", "catboost_cds_dt" in exp.models().index)
ets = exp.create_model("ets", cross_validation=False)          # the statsmodels-0.15 trap
best = exp.compare_models()                                     # every available model (turbo=True skips the slowest)
exp.plot_model(best, plot="forecast", data_kwargs={"fh": 24})
exp.predict_model(best)
final = exp.finalize_model(best)
exp.save_model(final, "colab_test_model"); _ = load_model("colab_test_model")
print()
print("ALL STEPS PASSED")

pycaret.__version__ = 3.5.0
pycaret.utils.version() = 3.5.0


,Number of airline passengers
Period,
1949-01,112.0
1949-02,118.0
1949-03,132.0
1949-04,129.0
1949-05,121.0


,Description,Value
0,session_id,42
1,Target,Number of airline passengers
2,Approach,Univariate
3,Exogenous Variables,Not Present
4,Original data shape,"(144, 1)"
5,Transformed data shape,"(144, 1)"
6,Transformed train set shape,"(132, 1)"
7,Transformed test set shape,"(12, 1)"
8,Rows with missing values,0.0%
9,Fold Generator,ExpandingWindowSplitter


models available: 29 | catboost_cds_dt: True


,MASE,RMSSE,MAE,RMSE,MAPE,SMAPE,R2
Test,0.3129,0.4475,9.5288,15.4600,0.0203,0.0199,0.9569


Processing:   0%|          | 0/4 [00:00<?, ?it/s]

,Model,MASE,RMSSE,MAE,RMSE,MAPE,SMAPE,R2,TT (Sec)
exp_smooth,Exponential Smoothing,0.5716,0.5997,16.7766,19.7952,0.0422,0.0427,0.8954,0.3067
ets,ETS,0.5929,0.6210,17.4105,20.5047,0.0440,0.0445,0.8883,0.4033
et_cds_dt,Extra Trees w/ Cond. Deseasonalize & Detrending,0.6666,0.7255,19.6620,24.0121,0.0490,0.0489,0.8465,1.1567
huber_cds_dt,Huber w/ Cond. Deseasonalize & Detrending,0.6813,0.7866,20.0334,25.9670,0.0491,0.0499,0.8113,0.8533
arima,ARIMA,0.6830,0.6735,20.0069,22.2199,0.0501,0.0507,0.8677,0.3400
lr_cds_dt,Linear w/ Cond. Deseasonalize & Detrending,0.7004,0.7702,20.6084,25.4401,0.0509,0.0514,0.8215,2.9767
ridge_cds_dt,Ridge w/ Cond. Deseasonalize & Detrending,0.7004,0.7703,20.6086,25.4405,0.0509,0.0514,0.8215,0.7167
en_cds_dt,Elastic Net w/ Cond. Deseasonalize & Detrending,0.7029,0.7732,20.6816,25.5362,0.0511,0.0516,0.8201,0.8867
lasso_cds_dt,Lasso w/ Cond. Deseasonalize & Detrending,0.7048,0.7751,20.7373,25.6005,0.0512,0.0517,0.8193,0.7233
llar_cds_dt,Lasso Least Angular Regressor w/ Cond. Deseasonalize & Detrending,0.7048,0.7751,20.7366,25.6009,0.0512,0.0517,0.8192,0.7267


Processing:   0%|          | 0/121 [00:00<?, ?it/s]

,Model,MASE,RMSSE,MAE,RMSE,MAPE,SMAPE,R2
0,Exponential Smoothing,0.3384,0.4576,10.3032,15.8104,0.0221,0.0216,0.9549


Transformation Pipeline and Model Successfully Saved
Transformation Pipeline and Model Successfully Loaded

ALL STEPS PASSED
